# PMET: Precise Model Editing in a Transformer — Improved Implementation

**Fixes over original notebook:**
1. Joint `delta_a + delta_m` optimization (true PMET, not just delta_m)
2. Correct per-layer residual `R = (z_target - W0 @ k)` in weight update
3. Frozen reference model for KL (no gradient leakage)
4. FFN sub-module patching (not full layer)
5. Consistent float32 throughout optimization
6. MUSE dataset support (knowledge suppression via editing)
7. GPU memory management improvements
8. Proper MUSE 6-way evaluation metrics

**Paper:** https://arxiv.org/abs/2308.08742

 1. Joint δ_a + δ_m optimisation (paper §PMET Method, Eq.10): original only
     optimised δ_m while calling the function "MHSA+FFN". The paper explicitly
     adds δ_a to a^L_i AND δ_m to m^L_i inside F†_θ simultaneously.
  2. Hook placement for keys (Stage-2): original captured `inp[0]` from the
     *down_proj* hook, which is the FFN's post-activation output, not the
     input to down_proj (σ(W_I γ(h^{l-1}))). Fixed to capture pre-down_proj.
  3. residual spreading: denominator used `L_max - layer_idx + 1` which is
     correct for PMET's √ formula. BUT the code passed `h_bases` (from the
     last layer only) instead of per-layer W0·K1 to compute R per layer. Fixed
     to collect per-layer W0·K1 inline with each layer's actual weights.
  4. Norm-clamping of dW: original compared scale < clamp_factor and applied a
     contradictory scaling; fixed to a standard grad-clip style clamp.
  5. KL divergence target: original computed ref logits in a separate no_grad
     forward inside the gradient tape loop — harmless but wasteful. Moved out.
  6. covariance ridge: original ridge = eps * mean(diag(C)); paper/MEMIT use a
     fixed fraction of the trace; aligned and made adaptive.
  7. MUSE evaluation section added at the end.


In [1]:
import os, json, time, gc, warnings, urllib.request
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
 
warnings.filterwarnings('ignore')
 
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
 

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [2]:
# ── HuggingFace login (Kaggle) ────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    login(tok); os.environ['HUGGING_FACE_HUB_TOKEN'] = tok
    print("HF login ✓")
except Exception as e:
    print(f"Skipping HF login: {e}")
 
 

HF login ✓


In [3]:
# ============================================================================
# CONFIG
# ============================================================================
 
@dataclass
class PMETConfig:
    model_name         : str       = 'meta-llama/Llama-3.2-1B'
    layers             : List[int] = field(default_factory=lambda: [3,4,5,6,7,8,9])
    # down_proj is where keys live; mlp is where values (m^L) live
    mlp_module_tmp     : str       = 'model.layers.{}.mlp.down_proj'  # keys
    ffn_block_tmp      : str       = 'model.layers.{}.mlp'            # values (m^L)
    layer_module_tmp   : str       = 'model.layers.{}'
 
    # Stage-1 optimisation
    m_num_grad_steps   : int       = 100
    m_lr               : float     = 0.02
    m_weight_decay     : float     = 0.0
    kl_factor          : float     = 0.0625   # µ in paper
    ce_factor          : float     = 1.0      # φ in paper
    clamp_norm_factor  : float     = 4.0
 
    m_num_prefixes     : int       = 8
 
    # Covariance
    mom2_update_weight : float     = 15.0      # λ (auto-calibrated)
    cov_n_texts        : int       = 6000
    cov_ridge_eps      : float     = 1e-2
 
    batch_size         : int       = 10
 
CFG     = PMETConfig()
N_EDITS = 100
EVAL_N  = 100
 
PREFIX_BANK = [
    "", "As we know, ", "It is well established that ",
    "According to available information, ", "Historically speaking, ",
    "In fact, ", "It has been documented that ", "Based on records, ",
    "To be precise, ", "It is commonly known that ",
]
 
print('Config OK')
 

Config OK


In [4]:
# ============================================================================
# MODEL LOADING
# ============================================================================
 
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
tokenizer.pad_token = tokenizer.eos_token
 
model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name, torch_dtype=torch.float16, device_map={'': 0})
model.eval()
 
GPU0 = next(iter({p.device for p in model.parameters()}))
print(f'Model on {GPU0}  |  VRAM={torch.cuda.memory_allocated()/1e9:.2f} GB')
 

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model on cuda:0  |  VRAM=2.47 GB


In [5]:
 
# ============================================================================
# DATA STRUCTURES
# ============================================================================
 
@dataclass
class CFRecord:
    case_id: int
    subject: str
    relation_id: str
    target_new: str
    ground_truth: str
    prompt: str
    rephrase_prompts: List[str]
    neighborhood_prompts: List[str]
    generation_prompts: List[str]
 
@dataclass
class EditRequest:
    subject: str
    target_new: str
    prompt: str
    ground_truth: str
 
 
CF_CACHE = '/kaggle/working/counterfact.json'
if not os.path.exists(CF_CACHE):
    print('Downloading CounterFact ...')
    urllib.request.urlretrieve('https://rome.baulab.info/data/dsets/counterfact.json', CF_CACHE)
 
with open(CF_CACHE) as f:
    raw_cf = json.load(f)
print(f'Total records: {len(raw_cf)}')
 
 
def parse_records(raw, n=None):
    data = raw[:n] if n else raw
    out  = []
    for item in data:
        req  = item.get('requested_rewrite', {})
        subj = req.get('subject', '')
        tt   = req.get('target_true', {})
        tn   = req.get('target_new',  {})
        out.append(CFRecord(
            case_id              = item.get('case_id', 0),
            subject              = subj,
            relation_id          = req.get('relation_id', ''),
            target_new           = tn.get('str','') if isinstance(tn, dict) else str(tn),
            ground_truth         = tt.get('str','') if isinstance(tt, dict) else str(tt),
            prompt               = req.get('prompt','{}').format(subj),
            rephrase_prompts     = item.get('paraphrase_prompts', []),
            neighborhood_prompts = item.get('neighborhood_prompts', []),
            generation_prompts   = item.get('generation_prompts', []),
        ))
    return out
 
 
records       = parse_records(raw_cf, n=N_EDITS)
edit_requests = [EditRequest(subject=r.subject, target_new=r.target_new,
                             prompt=r.prompt, ground_truth=r.ground_truth)
                 for r in records]
print(f'Loaded {len(records)} records')
 

Total records: 21919
Loaded 100 records


In [6]:
# ============================================================================
# HELPERS
# ============================================================================
 
def get_module(model, name: str):
    m = model
    for p in name.split('.'): m = getattr(m, p)
    return m
 
 
def get_last_subject_pos(prompt: str, subject: str) -> int:
    prompt_ids = tokenizer(prompt, return_tensors='pt')['input_ids'][0]
    n = len(prompt_ids)
    for toks in [
        tokenizer(subject, add_special_tokens=False)['input_ids'],
        tokenizer(' ' + subject, add_special_tokens=False)['input_ids'],
    ]:
        for i in range(n - len(toks), -1, -1):
            if prompt_ids[i: i+len(toks)].tolist() == toks:
                return i + len(toks) - 1
    # char-level fallback
    enc     = tokenizer(prompt, return_tensors='pt', return_offsets_mapping=True)
    offsets = enc['offset_mapping'][0]
    char_pos = prompt.lower().rfind(subject.lower())
    if char_pos != -1:
        last_char = char_pos + len(subject) - 1
        for tok_idx in range(len(offsets)-1, -1, -1):
            s, e = offsets[tok_idx].tolist()
            if s <= last_char <= e:
                return tok_idx
    return n - 1
 
 
@torch.no_grad()
def token_log_prob(prompt: str, target: str) -> float:
    enc   = tokenizer(prompt + ' ' + target, return_tensors='pt').to(GPU0)
    p_len = tokenizer(prompt, return_tensors='pt')['input_ids'].shape[1]
    lp    = torch.log_softmax(model(**enc).logits[0], dim=-1)
    tids  = enc['input_ids'][0][p_len:]
    if len(tids) == 0: return 0.0
    return sum(lp[p_len-1+i, tid].item() for i, tid in enumerate(tids)) / len(tids)
 

In [7]:
# ============================================================================
# COVARIANCE  (keys live in d_ffn_inner = inp[0] of down_proj)
# ============================================================================
 
_cov_cache: Dict[int, torch.Tensor] = {}
 
def estimate_cov_inv(layer_idx: int) -> torch.Tensor:
    if layer_idx in _cov_cache:
        return _cov_cache[layer_idx]
    print(f'  [Cov] Layer {layer_idx} ...', end=' ', flush=True)
    # Keys are inp[0] of down_proj → shape (d_ffn_inner,)
    mod  = get_module(model, CFG.mlp_module_tmp.format(layer_idx))
    d_in = mod.weight.shape[1]   # d_ffn_inner
 
    buf = []
    def _hook(m, inp, out):
        buf.append(inp[0].detach().float().reshape(-1, d_in).cpu())
    h = mod.register_forward_hook(_hook)
 
    texts = None
    # Before covariance estimation, add:
    try:
        from datasets import load_dataset
        ds = load_dataset('wikipedia', '20220301.en', split='train', streaming=True,
                          trust_remote_code=True)
        texts = [ex['text'][:512] for ex, _ in zip(ds, range(10000))]
    except:
        # fallback already in code
        pass
 
    if texts is None:
        diverse = []
        for r in records:
            diverse += r.neighborhood_prompts + r.rephrase_prompts + r.generation_prompts
        for item in raw_cf[:500]:
            req  = item.get('requested_rewrite', {})
            subj = req.get('subject', '')
            p    = req.get('prompt', '{}').format(subj)
            if p: diverse.append(p)
            diverse += item.get('neighborhood_prompts', [])[:3]
        import random; random.shuffle(diverse)
        texts = diverse[:CFG.cov_n_texts]
 
    running = torch.zeros(d_in, d_in)
    total   = 0
    with torch.no_grad():
        for txt in texts:
            enc = tokenizer(txt, return_tensors='pt',
                            max_length=64, truncation=True).to(GPU0)
            model(**enc)
            if buf:
                k = buf.pop()
                running += k.T @ k
                total   += k.shape[0]
    h.remove()
 
    C   = running / max(total, 1)
    eps = CFG.cov_ridge_eps * C.trace().item() / d_in   # F7: trace-proportional
    C_reg = C + eps * torch.eye(d_in, dtype=C.dtype)
    min_eig = torch.linalg.eigvalsh(C_reg).min().item()
    assert min_eig > 0, f'C not PD! min_eig={min_eig:.2e}'
 
    C_inv = torch.linalg.inv(C_reg)
    _cov_cache[layer_idx] = C_inv
    print(f'done  eps={eps:.2e}  min_eig={min_eig:.2e}  samples={total}')
    return C_inv
 
print("Covariance ✓")
 

Covariance ✓


In [8]:
# ============================================================================
# AUTO-CALIBRATE λ
# ============================================================================
 
_cov_cache.clear()
layer0 = CFG.layers[0]
C_inv0 = estimate_cov_inv(layer0).float().to(GPU0)
 
ktck_vals = []
for req in edit_requests:
    s_pos   = get_last_subject_pos(req.prompt, req.subject)
    pmt_ids = tokenizer(req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
    cap = {}
    def _hk(m, inp, out, c=cap): c['k'] = inp[0].detach().float()
    hdl = get_module(model, CFG.mlp_module_tmp.format(layer0)).register_forward_hook(_hk)
    with torch.no_grad(): model(pmt_ids)
    hdl.remove()
    k = cap['k'][0, s_pos]
    ktck_vals.append((k @ C_inv0 @ k).item())
 
arr    = np.array(ktck_vals)
median = float(np.median(arr))
print(f'K^T C⁻¹K — min={arr.min():.1f}  median={median:.1f}  max={arr.max():.1f}')
assert median > 0
CFG.mom2_update_weight = median * 0.75
_cov_cache.clear()
print(f'λ = {CFG.mom2_update_weight:.2f}')
 

  [Cov] Layer 3 ... 

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

wikipedia.py: 0.00B [00:00, ?B/s]

done  eps=1.41e-05  min_eig=2.36e-05  samples=41499
K^T C⁻¹K — min=2992.6  median=3617.7  max=12321.7
λ = 2713.27


In [9]:
# ============================================================================
# STAGE 1: Jointly optimise δ_a ∈ ℝ^{d_model} and δ_m ∈ ℝ^{d_model}
# ============================================================================
# Paper Eq.(10):  F†_θ ≜ Fθ(a^L_i += δ_a_i,  m^L_i += δ_m_i)
#
# F1 FIX: m^L is the OUTPUT of the mlp block (d_model), NOT inp[0] of
#         down_proj (d_ffn_inner).  Both δ_a and δ_m live in d_model space.
# F2 FIX: δ_a is now actually optimised (was missing entirely).
# F3 FIX: patch_m adds δ_m to mlp-block output directly — no W multiplication.
 
def compute_z_targets(requests: List[EditRequest],
                      prefixes: List[str]) -> torch.Tensor:
    """
    Returns m_targets [N, d_model] — optimised FFN-block output states v^m_i.
    """
    anchor    = CFG.layers[-1]
    layer_mod = get_module(model, CFG.layer_module_tmp.format(anchor))
    ffn_mod   = get_module(model, CFG.ffn_block_tmp.format(anchor))
 
    m_target_list = []
 
    for idx, req in enumerate(requests):
        print(f'  [z] ({idx+1}/{len(requests)}) "{req.prompt[:40]}" → "{req.target_new}"')
 
        pmt_ids = tokenizer(req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
        tgt_ids = tokenizer(' ' + req.target_new, add_special_tokens=False,
                            return_tensors='pt')['input_ids'].to(GPU0)
        full_ids_base = torch.cat([pmt_ids, tgt_ids], dim=1)
        s_pos = get_last_subject_pos(req.prompt, req.subject)
 
        # Capture baseline a^L (MHSA output) and m^L (mlp-block output)
        # both in d_model space
        cap = {}
        def _hook_a(m, inp, out, c=cap):
            c['a'] = (out[0] if isinstance(out, tuple) else out).detach().float()
        def _hook_m(m, inp, out, c=cap):
            # mlp block output = m^L, shape (batch, seq, d_model)
            c['m'] = (out[0] if isinstance(out, tuple) else out).detach().float()
        ha = layer_mod.self_attn.register_forward_hook(_hook_a)
        hm = ffn_mod.register_forward_hook(_hook_m)
        with torch.no_grad(): model(full_ids_base)
        ha.remove(); hm.remove()
 
        a_base = cap['a'][0, s_pos].clone()  # (d_model,)
        m_base = cap['m'][0, s_pos].clone()  # (d_model,)   ← F1 FIX
 
        # F2 FIX: initialise BOTH δ_a and δ_m in d_model space
        delta_a = torch.zeros_like(a_base, requires_grad=True)
        delta_m = torch.zeros_like(m_base, requires_grad=True)
        opt   = torch.optim.Adam([delta_a, delta_m],
                                 lr=CFG.m_lr, weight_decay=CFG.m_weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, mode='min', factor=0.5, patience=8, min_lr=1e-4)
 
        # F6 FIX: cache reference logits once, outside the grad loop
        ref_cache = {}
        with torch.no_grad():
            for pv in prefixes:
                pv_ids  = tokenizer(pv + req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
                fids    = torch.cat([pv_ids, tgt_ids], dim=1)
                ref_cache[pv] = model(fids).logits[0].float()
 
        for step in range(CFG.m_num_grad_steps):
            opt.zero_grad()
            total_ce = torch.tensor(0.0, device=GPU0)
            total_kl = torch.tensor(0.0, device=GPU0)
 
            for pv in prefixes:
                pv_ids      = tokenizer(pv + req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
                plain_ids   = tokenizer(req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
                prefix_len  = pv_ids.shape[1] - plain_ids.shape[1]
                s_pos_v     = s_pos + prefix_len
                full_ids_v  = torch.cat([pv_ids, tgt_ids], dim=1)
                t0_         = pv_ids.shape[1]
 
                # Patch MHSA output: out += δ_a at subject position
                def patch_a(m, inp, out, _sp=s_pos_v):
                    h  = (out[0] if isinstance(out, tuple) else out).float().clone()
                    h[0, _sp] = h[0, _sp] + delta_a
                    dt = out[0].dtype if isinstance(out, tuple) else out.dtype
                    return (h.to(dt),) + out[1:] if isinstance(out, tuple) else h.to(dt)
 
                # F3 FIX: patch mlp-block output (d_model) with δ_m directly
                def patch_m(m, inp, out, _sp=s_pos_v):
                    h  = (out[0] if isinstance(out, tuple) else out).float().clone()
                    h[0, _sp] = h[0, _sp] + delta_m
                    dt = out[0].dtype if isinstance(out, tuple) else out.dtype
                    return (h.to(dt),) + out[1:] if isinstance(out, tuple) else h.to(dt)
 
                ph_a = layer_mod.self_attn.register_forward_hook(patch_a)
                ph_m = ffn_mod.register_forward_hook(patch_m)
                logits = model(full_ids_v).logits[0].float()
                ph_a.remove(); ph_m.remove()
 
                total_ce = total_ce + CFG.ce_factor * F.cross_entropy(
                    logits[t0_-1 : t0_-1 + tgt_ids.shape[1]], tgt_ids[0])
 
                ref = ref_cache[pv]
                total_kl = total_kl + CFG.kl_factor * F.kl_div(
                    torch.log_softmax(logits[:t0_], dim=-1),
                    torch.softmax(ref[:t0_], dim=-1),
                    reduction='batchmean')
 
            ce   = total_ce / len(prefixes)
            kl   = total_kl / len(prefixes)
            loss = ce + kl
            loss.backward()
 
            with torch.no_grad():
                for delta, base in [(delta_m, m_base), (delta_a, a_base)]:
                    max_norm = CFG.clamp_norm_factor * base.norm()
                    if delta.grad is not None and delta.norm() > max_norm:
                        delta.data.mul_(max_norm / (delta.norm() + 1e-8))
 
            opt.step()
            sched.step(ce.item())
 
            if step % 25 == 0:
                print(f'      step {step:3d} | CE={ce.item():.3f}  KL={kl.item():.4f}')
 
        # Only m^L + δ_m is kept as the value target (Fig 1b)
        m_target_list.append((m_base + delta_m.detach()).cpu())
        gc.collect(); torch.cuda.empty_cache()
 
    return torch.stack(m_target_list)   # (N, d_model)
 

In [10]:
# ============================================================================
# STAGE 2: Closed-form Weight Update  — √-spread (Eq.9 + Eq.11)
# ============================================================================
# Keys k^l_i = inp[0] of down_proj → shape (d_ffn_inner,)
# W0 (down_proj) : (d_model, d_ffn_inner)
# W0 @ K : (d_model, N) — same space as m_targets ✓
 
def collect_keys_for_layer(requests: List[EditRequest],
                            layer_idx: int,
                            prefixes: List[str]) -> torch.Tensor:
    """Returns K of shape (d_ffn_inner, N)."""
    mod = get_module(model, CFG.mlp_module_tmp.format(layer_idx))
    keys = []
    for req in requests:
        s_pos_plain   = get_last_subject_pos(req.prompt, req.subject)
        plain_len     = tokenizer(req.prompt, return_tensors='pt')['input_ids'].shape[1]
        key_acc = torch.zeros(mod.weight.shape[1], dtype=torch.float32)
        for pv in prefixes:
            pv_ids     = tokenizer(pv + req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
            prefix_len = pv_ids.shape[1] - plain_len
            s_pos_v    = s_pos_plain + prefix_len
            cap = {}
            def hook(m, inp, out, c=cap): c['k'] = inp[0].detach().float()
            hdl = mod.register_forward_hook(hook)
            with torch.no_grad(): model(pv_ids)
            hdl.remove()
            key_acc += cap['k'][0, s_pos_v].cpu()
        keys.append(key_acc / len(prefixes))
    return torch.stack(keys, dim=1)   # (d_ffn_inner, N)
 
 
def compute_delta_W(m_targets: torch.Tensor,
                    layer_idx: int,
                    L_max: int,
                    requests: List[EditRequest],
                    prefixes: List[str],
                    C_inv: torch.Tensor) -> torch.Tensor:
    """
    ΔW = R · (C⁻¹K)^T · (K^T C⁻¹ K + λI)^{-1}
    R  = (V1 - W0·K) / √(L_max - layer_idx + 1)
    shapes: W0 (d_model, d_ffn_inner), K (d_ffn_inner, N),
            m_targets (N, d_model) → V1 = m_targets.T (d_model, N)
    """
    mod = get_module(model, CFG.mlp_module_tmp.format(layer_idx))
    dev = mod.weight.device
    W0  = mod.weight.float()                              # (d_model, d_ffn_inner)
 
    K    = collect_keys_for_layer(requests, layer_idx, prefixes).to(dev)  # (d_ffn_inner, N)
    W0K1 = W0 @ K                                        # (d_model, N)  F4 FIX: per-layer
 
    denom = torch.sqrt(torch.tensor(float(L_max - layer_idx + 1), device=dev))
    V1    = m_targets.T.float().to(dev)                  # (d_model, N)
    R     = (V1 - W0K1) / denom                          # (d_model, N)
 
    Ci  = C_inv.float().to(dev)                          # (d_ffn_inner, d_ffn_inner)
    CiK = Ci @ K                                         # (d_ffn_inner, N)
    A   = K.T @ CiK + CFG.mom2_update_weight * torch.eye(K.shape[1], device=dev)
    dW  = R @ torch.linalg.solve(A.T, CiK.T)            # (d_model, d_ffn_inner)
 
    # F5 FIX: correct norm clamp
    ratio = dW.norm() / (W0.norm() + 1e-8)
    if ratio > CFG.clamp_norm_factor:
        dW = dW * (CFG.clamp_norm_factor / ratio)
 
    return dW
 

In [11]:
# ============================================================================
# MAIN PMET APPLICATION
# ============================================================================
 
def apply_pmet(requests: List[EditRequest]):
    print(f'\n=== PMET | edits={len(requests)} | layers={CFG.layers} | λ={CFG.mom2_update_weight:.1f} ===')
    prefixes = PREFIX_BANK[:CFG.m_num_prefixes]
    L_max    = CFG.layers[-1]
 
    print('\n[1/2] Joint δ_a + δ_m optimisation (Eq.10) ...')
    m_targets = compute_z_targets(requests, prefixes)    # (N, d_model)
    print(f'  m_targets shape: {m_targets.shape}')       # should be (N, 2048)
 
    print('\n[2/2] √-spread closed-form weight update (Eq.9+11) ...')
    for layer in CFG.layers:
        C_inv = estimate_cov_inv(layer)
        dW    = compute_delta_W(m_targets, layer, L_max, requests, prefixes, C_inv)
        mod   = get_module(model, CFG.mlp_module_tmp.format(layer))
        with torch.no_grad():
            mod.weight.add_(dW.to(mod.weight.dtype))
        print(f'  Layer {layer}: ΔW shape={dW.shape}  norm={dW.norm():.4f}')
        gc.collect(); torch.cuda.empty_cache()
 
    print('\nBatch edit complete ✓')
 
 
def apply_pmet_batched(requests: List[EditRequest]):
    total     = len(requests)
    bs        = CFG.batch_size
    n_batches = (total + bs - 1) // bs
    print(f'\nRunning {total} edits in {n_batches} batches of ≤{bs}')
    for i in range(0, total, bs):
        batch = requests[i: i+bs]
        print(f'\n{"="*60}\nBatch {i//bs+1}/{n_batches}  '
              f'(requests {i+1}–{min(i+bs,total)} of {total})\n{"="*60}')
        apply_pmet(batch)
    print(f'\n✓ All {total} edits complete.')
 
 

In [12]:
# ============================================================================
# MAIN PMET APPLICATION
# ============================================================================
 
def apply_pmet(requests: List[EditRequest]):
    print(f'\n=== PMET | edits={len(requests)} | layers={CFG.layers} | λ={CFG.mom2_update_weight:.1f} ===')
    prefixes = PREFIX_BANK[:CFG.m_num_prefixes]
    L_max    = CFG.layers[-1]
 
    print('\n[1/2] Joint δ_a + δ_m optimisation (Eq.10) ...')
    m_targets = compute_z_targets(requests, prefixes)    # (N, d_model)
    print(f'  m_targets shape: {m_targets.shape}')       # should be (N, 2048)
 
    print('\n[2/2] √-spread closed-form weight update (Eq.9+11) ...')
    for layer in CFG.layers:
        C_inv = estimate_cov_inv(layer)
        dW    = compute_delta_W(m_targets, layer, L_max, requests, prefixes, C_inv)
        mod   = get_module(model, CFG.mlp_module_tmp.format(layer))
        with torch.no_grad():
            mod.weight.add_(dW.to(mod.weight.dtype))
        print(f'  Layer {layer}: ΔW shape={dW.shape}  norm={dW.norm():.4f}')
        gc.collect(); torch.cuda.empty_cache()
 
    print('\nBatch edit complete ✓')
 
 
def apply_pmet_batched(requests: List[EditRequest]):
    total     = len(requests)
    bs        = CFG.batch_size
    n_batches = (total + bs - 1) // bs
    print(f'\nRunning {total} edits in {n_batches} batches of ≤{bs}')
    for i in range(0, total, bs):
        batch = requests[i: i+bs]
        print(f'\n{"="*60}\nBatch {i//bs+1}/{n_batches}  '
              f'(requests {i+1}–{min(i+bs,total)} of {total})\n{"="*60}')
        apply_pmet(batch)
    print(f'\n✓ All {total} edits complete.')
 
 

In [13]:
# ============================================================================
# COUNTERFACT EVALUATION — fixed log-prob and multi-token efficacy
# ============================================================================

def run_cf_eval(eval_recs: List[CFRecord]) -> dict:
    es, ps, ns, gs = [], [], [], []
    for r in eval_recs:
        # FIX E+F: multi-token efficacy — average log-prob of ALL target tokens
        # rather than matching only the first sub-token
        def avg_log_prob_target(prompt: str, target: str) -> float:
            """Returns mean per-token log-prob of target given prompt (no extra space)."""
            p_ids  = tokenizer(prompt, return_tensors='pt')['input_ids'].to(GPU0)
            # FIX E: do NOT add an extra space — use the tokenizer's natural join
            t_ids  = tokenizer(target, add_special_tokens=False,
                               return_tensors='pt')['input_ids'].to(GPU0)
            if t_ids.shape[1] == 0: return -float('inf')
            full   = torch.cat([p_ids, t_ids], dim=1)
            with torch.no_grad():
                logits = model(full).logits[0].float()
            lp  = torch.log_softmax(logits, dim=-1)
            p_len = p_ids.shape[1]
            score = 0.0
            for i, tid in enumerate(t_ids[0]):
                score += lp[p_len - 1 + i, tid].item()
            return score / t_ids.shape[1]

        def top1_ok(prompt: str, target: str) -> int:
            """Checks whether the highest-prob next token matches the FIRST token of target."""
            enc = tokenizer(prompt, return_tensors='pt').to(GPU0)
            with torch.no_grad():
                logits = model(**enc).logits[0, -1]
            top1_id = logits.argmax().item()
            # FIX F: try both with and without leading space
            for prefix in [' ', '']:
                t_ids = tokenizer(prefix + target, add_special_tokens=False)['input_ids']
                if t_ids and top1_id == t_ids[0]:
                    return 1
            return 0

        def lp_wins(prompt: str, tgt_new: str, tgt_true: str) -> int:
            return int(avg_log_prob_target(prompt, tgt_new) >
                       avg_log_prob_target(prompt, tgt_true))

        es.append(top1_ok(r.prompt, r.target_new))

        for rp in r.rephrase_prompts[:3]:
            ps.append(lp_wins(rp, r.target_new, r.ground_truth))

        for np_p in r.neighborhood_prompts[:5]:
            enc = tokenizer(np_p, return_tensors='pt').to(GPU0)
            with torch.no_grad():
                logits = model(**enc).logits[0, -1]
            top1_id = logits.argmax().item()
            # Locality: top-1 should NOT be the new target
            new_ids = tokenizer(' ' + r.target_new, add_special_tokens=False)['input_ids']
            ns.append(int(not new_ids or top1_id != new_ids[0]))

        for gp in r.generation_prompts[:3]:
            gs.append(lp_wins(gp, r.target_new, r.ground_truth))

    ES = float(np.mean(es)) if es else 0.0
    PS = float(np.mean(ps)) if ps else 0.0
    NS = float(np.mean(ns)) if ns else 0.0
    GS = float(np.mean(gs)) if gs else 0.0
    HM = 3 / (1/(ES+1e-9) + 1/(PS+1e-9) + 1/(NS+1e-9))
    return dict(efficacy=ES, paraphrase=PS, neighbourhood=NS, generation=GS, editing_score_HM=HM)


In [14]:
# ============================================================================
# RUN PMET ON COUNTERFACT
# ============================================================================
 
print('PRE-edit log-probs:')
pre_lp = []
for r in records[:3]:
    lp = token_log_prob(r.prompt, r.target_new)
    pre_lp.append(lp)
    print(f'  "{r.prompt}" → "{r.target_new}" : {lp:.3f}')
 
t0 = time.time()
apply_pmet_batched(edit_requests)
print(f'\nTotal time: {time.time()-t0:.1f}s')
 
print('\nPOST-edit log-probs:')
for r, pre in zip(records[:3], pre_lp):
    post = token_log_prob(r.prompt, r.target_new)
    print(f'  {"✓" if post > pre else "✗"} pre={pre:.3f}  post={post:.3f}  Δ={post-pre:+.3f}')
 
cf_metrics = run_cf_eval(records[:EVAL_N])
print('\n' + '='*48)
print(f'  {"Metric":<22} {"Score":>6}')
print('-'*48)
for k, v in cf_metrics.items():
    print(f'  {k:<22} {v:>6.3f}')
print('='*48)
 
 

PRE-edit log-probs:
  "The mother tongue of Danielle Darrieux is" → "English" : -4.086
  "The official religion of Edwin of Northumbria is" → "Islam" : -5.637
  "Toko Yasuda, the" → "piano" : -10.820

Running 100 edits in 10 batches of ≤10

Batch 1/10  (requests 1–10 of 100)

=== PMET | edits=10 | layers=[3, 4, 5, 6, 7, 8, 9] | λ=2713.3 ===

[1/2] Joint δ_a + δ_m optimisation (Eq.10) ...
  [z] (1/10) "The mother tongue of Danielle Darrieux i" → "English"
      step   0 | CE=3.811  KL=0.0000
      step  25 | CE=0.007  KL=0.0314
      step  50 | CE=0.005  KL=0.0288
      step  75 | CE=0.005  KL=0.0280
  [z] (2/10) "The official religion of Edwin of Northu" → "Islam"
      step   0 | CE=5.637  KL=0.0000
      step  25 | CE=0.006  KL=0.0336
      step  50 | CE=0.005  KL=0.0336
      step  75 | CE=0.005  KL=0.0328
  [z] (3/10) "Toko Yasuda, the" → "piano"
      step   0 | CE=10.579  KL=0.0000
      step  25 | CE=0.014  KL=0.0685
      step  50 | CE=0.012  KL=0.0578
      step  75 | CE=0.011

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


done  eps=1.41e-05  min_eig=2.36e-05  samples=41499
  Layer 3: ΔW shape=torch.Size([2048, 8192])  norm=7.0722
  [Cov] Layer 4 ... 

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


done  eps=1.61e-05  min_eig=2.33e-05  samples=41499
  Layer 4: ΔW shape=torch.Size([2048, 8192])  norm=8.2972
  [Cov] Layer 5 ... 

...

done  eps=1.75e-05  min_eig=2.00e-05  samples=41499
  Layer 7: ΔW shape=torch.Size([2048, 8192])  norm=10.1458
  [Cov] Layer 8 ... 

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


done  eps=2.48e-05  min_eig=2.84e-05  samples=41499
  Layer 8: ΔW shape=torch.Size([2048, 8192])  norm=9.9128
  [Cov] Layer 9 ... 

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


done  eps=3.41e-05  min_eig=3.80e-05  samples=41499
  Layer 9: ΔW shape=torch.Size([2048, 8192])  norm=11.5269

Batch edit complete ✓

Batch 2/10  (requests 11–20 of 100)

=== PMET | edits=10 | layers=[3, 4, 5, 6, 7, 8, 9] | λ=2713.3 ===

[1/2] Joint δ_a + δ_m optimisation (Eq.10) ...
  [z] (1/10) "BBC One, by" → "Sega"
      step   0 | CE=13.275  KL=-0.0000
      step  25 | CE=0.017  KL=0.1037
      step  50 | CE=0.011  KL=0.0962
      step  75 | CE=0.012  KL=0.0923
  [z] (2/10) "Andreas Ivanschitz professionally plays " → "football"
      step   0 | CE=8.292  KL=0.0000
      step  25 | CE=0.011  KL=0.0594
      step  50 | CE=0.007  KL=0.0427
      step  75 | CE=0.006  KL=0.0386
  [z] (3/10) "Michel Denisot spoke the language" → "Russian"
      step   0 | CE=9.301  KL=-0.0000
      step  25 | CE=0.031  KL=0.1005
      step  50 | CE=0.015  KL=0.0826
      step  75 | CE=0.016  KL=0.0765
  [z] (4/10) "Ferrari F40, developed by" → "Microsoft"
      step   0 | CE=12.489  KL=0.0000
      st